# Ropedia Academy — D3 · TSDF fusion & submaps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D3.ipynb)

> **Runs TSDF fusion over 50 noisy depth frames and plots the truncated field sharpening to a clean zero-crossing (the surface) as frames accumulate.**
>
> 对 50 帧含噪深度做 TSDF 融合，并画出随帧累积、截断场收敛到干净零交叉（表面）的过程。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D3

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(0)

# TSDF fusion: average many NOISY depth frames into one clean surface.
voxels = np.linspace(0, 2, 81); trunc = 0.2
tsdf = np.zeros_like(voxels); weight = np.zeros_like(voxels)
snapshots = []
for i in range(50):                               # 50 noisy depth measurements
    depth = 1.0 + np.random.randn()*0.05
    sdf = np.clip((depth - voxels)/trunc, -1, 1)  # TRUNCATED signed distance
    near = np.abs(depth - voxels) < trunc
    tsdf[near] = (weight[near]*tsdf[near] + sdf[near]) / (weight[near]+1)   # weighted avg
    weight[near] += 1
    if i in (0, 4, 49): snapshots.append((i+1, tsdf.copy()))

zc = np.where(np.diff(np.sign(tsdf[weight>0])))[0]
print("fused surface depth:", voxels[weight>0][zc].round(3), "(true 1.0)")

# Visualize the TSDF converging to a clean zero-crossing as frames accumulate.
plt.figure(figsize=(6, 3.3))
for n, snap in snapshots:
    plt.plot(voxels[weight>0], snap[weight>0], '-o', ms=2, label=f"{n} frame(s)")
plt.axhline(0, c='gray', ls='--'); plt.axvline(1.0, c='g', ls=':', label='true surface')
plt.title("TSDF fusion: noise averages out to a clean surface")
plt.xlabel("depth"); plt.ylabel("TSDF"); plt.legend(fontsize=8); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D3
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks